In [1]:
import pandas as pd

# No additional imports are needed; you already imported 'pandas as pd' above.
data_dir = "/Users/davidabelson/Library/CloudStorage/OneDrive-UniversityofCambridge/Aaron Weimann's files - project_k/data/final/metadata/"
metadata_file = '/Users/davidabelson/Library/CloudStorage/OneDrive-UniversityofCambridge/Code_repos/md_curation/metadata_final_curated_all_samples_and_columns.tsv'
output_dir = data_dir + "/final/"

metadata = pd.read_csv(metadata_file, sep="\t", low_memory=False)

In [2]:
# Next, we want to check long reads lr = metadata['is_refseq']
lr = metadata[metadata['is_refseq']]
print(f"Number of long reads: {len(lr)}")

# sample_alias
print("*"*80)
print("sample_alias")
print(lr['sample_alias'].head(3))
print("*"*80)

# Check columns Sample, sample_accession, BioSample
for col in metadata.columns:
    if '_accession' in col:
        print(f"Column: {col}")
        #print first 3 values
        print(metadata[col].head(3))

for col in metadata.columns:
    if 'sample' in col and '_accession' not in col:
        print(f"Column: {col}")
        print(metadata[col].head(3))




Number of long reads: 3513
********************************************************************************
sample_alias
82860    NaN
82861    NaN
82862    NaN
Name: sample_alias, dtype: object
********************************************************************************
Column: metadata.sample.secondary_accession
0    DRS014084
1    DRS013791
2    DRS013790
Name: metadata.sample.secondary_accession, dtype: object
Column: metadata.studies.secondary_accession
0    DRP001650
1    DRP001367
2    DRP001366
Name: metadata.studies.secondary_accession, dtype: object
Column: metadata.runs.submission_accession
0                 DRA001578
1    DRA001296 || DRA001296
2    DRA001295 || DRA001295
Name: metadata.runs.submission_accession, dtype: object
Column: sample_accession
0    SAMD00002693
1    SAMD00008836
2    SAMD00008944
Name: sample_accession, dtype: object
Column: run_accession
0              DRR015976
1    DRR015584,DRR015583
2    DRR015581,DRR015582
Name: run_accession, dtype: object

In [11]:
# ──────────────────────────────────────────────────────────────────────────
# Diagnostic: of the long-reads find_long_reads.py wrote into
# related_lr_accession, how many were already implied by the metadata
# itself (so the BioSample lookup was redundant) vs genuinely new info?
#
# Population analysed:
#   metadata['is_refseq'] == False
#   AND metadata['kpsc_final_list'] == True
#   AND metadata['related_lr_accession'].notna()
#   → 2,231 rows. These are KPSC short-read samples for which the script
#     wrote a long-read accession into related_lr_accession.
#
# NOTE: find_long_reads.py itself does NOT apply kpsc_final_list to its
# non-RefSeq branch (find_long_reads.py:1581) — so the raw TSV contains
# 3,499 populated rows including 1,268 non-KPSC. We filter here.
#
# Two independent definitions of "the lookup was redundant":
#   A) The row's own platform field (instrument_platform OR
#      metadata.runs.instrument.platform) already mentioned ONT/PacBio.
#   B) The specific run accession written to related_lr_accession was
#      already listed in the row's run_accession field (comma-separated).
# ──────────────────────────────────────────────────────────────────────────

LR_TOKENS = ("OXFORD_NANOPORE", "PACBIO_SMRT")

def _has_lr_token(val):
    if not isinstance(val, str):
        return False
    return any(tok in val for tok in LR_TOKENS)

sub = metadata[
    (metadata['is_refseq'] == False)
    & (metadata['kpsc_final_list'] == True)
    & metadata['related_lr_accession'].notna()
].copy()
n_sub = len(sub)
n_pop = int(((metadata['is_refseq'] == False) & (metadata['kpsc_final_list'] == True)).sum())

# Definition A — platform field already mentions ONT/PacBio
plat_in_summary = sub['instrument_platform'].apply(_has_lr_token)
plat_in_perrun = sub['metadata.runs.instrument.platform'].apply(_has_lr_token)
platform_already_flagged_lr = plat_in_summary | plat_in_perrun

# Split related_lr_accession into "ENA run" (DRR/SRR/ERR…) vs "GCF assembly"
is_gcf = sub['related_lr_accession'].str.startswith('GCF_', na=False)

# Definition B — found run accession already listed in row's run_accession
def _run_already_listed(row):
    acc = row['related_lr_accession']
    if not isinstance(acc, str) or acc.startswith('GCF_'):
        return None  # GCF assembly accessions can't be matched against run_accession
    existing = row.get('run_accession')
    if not isinstance(existing, str):
        return False
    existing_runs = {x.strip() for x in existing.split(',')}
    return acc in existing_runs

sub['run_already_in_run_accession'] = sub.apply(_run_already_listed, axis=1)
sub['platform_already_flagged_lr'] = platform_already_flagged_lr

# ── Print summary ──
print("=" * 78)
print(f"Population: is_refseq=False & kpsc_final_list=True rows     {n_pop:>8,}")
print(f"  ↳ with related_lr_accession populated (analysed below):   {n_sub:>8,}")
print()
print("Breakdown of related_lr_accession type:")
print(f"  ENA run-style (DRR/SRR/ERR…):                             {(~is_gcf).sum():>8,}")
print(f"  GCF assembly accession:                                   {is_gcf.sum():>8,}")
print()
print("=" * 78)
print("Definition A — did the row's own platform field already mention ONT/PacBio?")
print("(checks instrument_platform AND metadata.runs.instrument.platform)")
print("-" * 78)
n_known_a = int(platform_already_flagged_lr.sum())
n_novel_a = n_sub - n_known_a
print(f"  platform already mentioned ONT/PacBio (lookup redundant): {n_known_a:>8,}  ({n_known_a / n_sub:.1%})")
print(f"  platform was Illumina-only (lookup added information):    {n_novel_a:>8,}  ({n_novel_a / n_sub:.1%})")
print()
print("=" * 78)
print("Definition B — was the found run accession already in run_accession?")
print("(only ENA run-style matches; GCF assembly accessions excluded)")
print("-" * 78)
runs_only = sub[~is_gcf]
n_redundant_b = int((runs_only['run_already_in_run_accession'] == True).sum())
n_novel_b = int((runs_only['run_already_in_run_accession'] == False).sum())
denom = max(len(runs_only), 1)
print(f"  run was already in run_accession (lookup redundant):      {n_redundant_b:>8,}  ({n_redundant_b / denom:.1%})")
print(f"  run was NOT in run_accession (lookup added information):  {n_novel_b:>8,}  ({n_novel_b / denom:.1%})")
print(f"  (GCF assembly matches, not applicable to definition B):   {int(is_gcf.sum()):>8,}")
print()
print("=" * 78)
print("Cross-tab (A × B), ENA run-style matches only")
print("-" * 78)
ctab = pd.crosstab(
    runs_only['platform_already_flagged_lr'].map({
        True:  'platform already flagged LR',
        False: 'platform Illumina-only',
    }),
    runs_only['run_already_in_run_accession'].map({
        True:  'run already in run_accession',
        False: 'run NOT in run_accession',
    }),
    margins=True,
    margins_name='total',
)
print(ctab.to_string())
print()

# ── Example rows ──
def _show_examples(df, label, n=5):
    cols = ['sample_accession', 'run_accession', 'related_lr_accession',
            'instrument_platform', 'metadata.runs.instrument.platform']
    cols = [c for c in cols if c in df.columns]
    print("=" * 78)
    print(f"Examples — {label} (up to {n} rows):")
    print("-" * 78)
    if len(df) == 0:
        print("  (none)")
    else:
        print(df[cols].head(n).to_string(index=False))
    print()

_show_examples(sub[~platform_already_flagged_lr],
               "Def A novel: platform was Illumina-only, lookup found LR")
_show_examples(runs_only[runs_only['run_already_in_run_accession'] == False],
               "Def B novel: found run was NOT previously in run_accession")
_show_examples(runs_only[runs_only['run_already_in_run_accession'] == True],
               "Def B redundant: found run was already listed in run_accession")

Population: is_refseq=False & kpsc_final_list=True rows       75,715
  ↳ with related_lr_accession populated (analysed below):      2,231

Breakdown of related_lr_accession type:
  ENA run-style (DRR/SRR/ERR…):                                2,231
  GCF assembly accession:                                          0

Definition A — did the row's own platform field already mention ONT/PacBio?
(checks instrument_platform AND metadata.runs.instrument.platform)
------------------------------------------------------------------------------
  platform already mentioned ONT/PacBio (lookup redundant):    1,161  (52.0%)
  platform was Illumina-only (lookup added information):       1,070  (48.0%)

Definition B — was the found run accession already in run_accession?
(only ENA run-style matches; GCF assembly accessions excluded)
------------------------------------------------------------------------------
  run was already in run_accession (lookup redundant):           222  (10.0%)
  run was NOT 

In [13]:
# ──────────────────────────────────────────────────────────────────────────
# Assembly quality of the "hybrid" rows
# (Def A "platform already mentioned ONT/PacBio" subset of the KPSC,
# is_refseq=False, related_lr_accession-populated population.)
#
# Reports: n_contigs and N50-as-%-of-genome — distribution stats that
# indicate how close these assemblies are to closed/complete genomes.
# Uses bakta.stats.* columns. NOTE these reflect the *short-read* (Illumina)
# assembly on this row, not a hybrid assembly.
# ──────────────────────────────────────────────────────────────────────────

def _has_lr_token(val):
    if not isinstance(val, str):
        return False
    return any(tok in val for tok in ("OXFORD_NANOPORE", "PACBIO_SMRT"))

sub = metadata[
    (metadata['is_refseq'] == False)
    & (metadata['kpsc_final_list'] == True)
    & metadata['related_lr_accession'].notna()
].copy()
hybrid = sub[
    sub['instrument_platform'].apply(_has_lr_token)
    | sub['metadata.runs.instrument.platform'].apply(_has_lr_token)
].copy()

n_contigs = hybrid['bakta.stats.no_sequences']
n50 = hybrid['bakta.stats.n50']
genome_size = hybrid['bakta.stats.size']
n50_pct = (n50 / genome_size) * 100.0

def _stats(s, label):
    q1, med, q3 = s.quantile([0.25, 0.5, 0.75])
    print(f"  {label}")
    print(f"    n           : {s.notna().sum():>12,}")
    print(f"    min         : {('{:>12,.0f}'.format(s.min()) if pd.notna(s.min()) else 'n/a')}")
    print(f"    Q1          : {('{:>12,.0f}'.format(q1) if pd.notna(q1) else 'n/a')}")
    print(f"    median      : {('{:>12,.0f}'.format(med) if pd.notna(med) else 'n/a')}")
    print(f"    Q3          : {('{:>12,.0f}'.format(q3) if pd.notna(q3) else 'n/a')}")
    print(f"    max         : {('{:>12,.0f}'.format(s.max()) if pd.notna(s.max()) else 'n/a')}")
    print(f"    IQR         : {('{:>12,.0f}'.format(q3 - q1) if pd.notna(q1) and pd.notna(q3) else 'n/a')}")


print("=" * 78)
print(f"Hybrid subset (KPSC, is_refseq=False, Def A 'platform already ONT/PacBio')")
print(f"n = {len(hybrid):,} rows")
print("Assembly stats from bakta.stats.* (no_sequences, n50, size)")
print("=" * 78)
print()
_stats(n_contigs, "Number of contigs (bakta.stats.no_sequences)")
print()

# Bonus: how many look effectively closed?
print("=" * 78)
print("Reference points — how close to closed are these (short-read) assemblies?")
print("-" * 78)
print(f"  n_contigs ≤ 1                    : {int((n_contigs <= 1).sum()):>5,}  "
      f"({(n_contigs <= 1).mean():.1%})   (single contig — chromosome only or fully closed)")
print(f"  n_contigs ≤ 5                    : {int((n_contigs <= 5).sum()):>5,}  "
      f"({(n_contigs <= 5).mean():.1%})   (chromosome + a few plasmids)")
print(f"  n_contigs ≤ 10                   : {int((n_contigs <= 10).sum()):>5,}  "
      f"({(n_contigs <= 10).mean():.1%})")
print(f"  N50 ≥ 50% of genome              : {int((n50_pct >= 50).sum()):>5,}  "
      f"({(n50_pct >= 50).mean():.1%})   (≈chromosome-scale contig present)")
print(f"  N50 ≥ 90% of genome              : {int((n50_pct >= 90).sum()):>5,}  "
      f"({(n50_pct >= 90).mean():.1%})   (essentially closed chromosome)")

Hybrid subset (KPSC, is_refseq=False, Def A 'platform already ONT/PacBio')
n = 1,161 rows
Assembly stats from bakta.stats.* (no_sequences, n50, size)

  Number of contigs (bakta.stats.no_sequences)
    n           :        1,161
    min         :           34
    Q1          :          100
    median      :          131
    Q3          :          174
    max         :          359
    IQR         :           74

Reference points — how close to closed are these (short-read) assemblies?
------------------------------------------------------------------------------
  n_contigs ≤ 1                    :     0  (0.0%)   (single contig — chromosome only or fully closed)
  n_contigs ≤ 5                    :     0  (0.0%)   (chromosome + a few plasmids)
  n_contigs ≤ 10                   :     0  (0.0%)
  N50 ≥ 50% of genome              :     2  (0.2%)   (≈chromosome-scale contig present)
  N50 ≥ 90% of genome              :     0  (0.0%)   (essentially closed chromosome)


In [15]:
# ──────────────────────────────────────────────────────────────────────────
# Of the related_lr_accession rows (KPSC + is_refseq=False), how many have
# *enough* long-read coverage to actually do a hybrid assembly?
#
# `sufficient_for_hybrid` and `est_coverage_5_5Mb` are the final two columns
# of the metadata. NOTE: `sufficient_for_hybrid` lives on the LONG-READ row
# (the row whose run_accession == related_lr_accession), not on the
# short-read row — we look it up.
#
# Subgroups (from Definition A):
#   Ai  = "platform already mentioned ONT/PacBio" (hybrid candidates already
#         flagged in the metadata)
#   Aii = "platform was Illumina-only" (long-reads only discovered via the
#         BioSample lookup)
# ──────────────────────────────────────────────────────────────────────────

def _has_lr_token(v):
    return isinstance(v, str) and any(t in v for t in ("OXFORD_NANOPORE", "PACBIO_SMRT"))

sub = metadata[
    (metadata['is_refseq'] == False)
    & (metadata['kpsc_final_list'] == True)
    & metadata['related_lr_accession'].notna()
].copy()
sub['platform_flagged_lr'] = (
    sub['instrument_platform'].apply(_has_lr_token)
    | sub['metadata.runs.instrument.platform'].apply(_has_lr_token)
)

# Look up the long-read row by matching related_lr_accession against run_accession
run_to_idx = {}
for idx, ra in metadata['run_accession'].items():
    if isinstance(ra, str):
        for r in ra.split(','):
            run_to_idx[r.strip()] = idx

def _lookup(acc):
    if not isinstance(acc, str) or acc.startswith('GCF_'):
        return None
    return run_to_idx.get(acc)

sub['lr_row_idx'] = sub['related_lr_accession'].apply(_lookup)
lr_idx = sub['lr_row_idx'].dropna().astype(int)
sub.loc[lr_idx.index, 'lr_sufficient_for_hybrid'] = metadata.loc[lr_idx.values, 'sufficient_for_hybrid'].values
sub.loc[lr_idx.index, 'lr_est_coverage_5_5Mb'] = metadata.loc[lr_idx.values, 'est_coverage_5_5Mb'].values

# For any GCF assembly matches (expected: 0 after kpsc filter), use est_coverage
# on the short-read row itself.
gcf_mask = sub['related_lr_accession'].str.startswith('GCF_', na=False)
sub.loc[gcf_mask, 'lr_est_coverage_5_5Mb'] = sub.loc[gcf_mask, 'est_coverage_5_5Mb']

def _report(group_df, group_label):
    print("=" * 78)
    print(f"{group_label}   (n = {len(group_df):,})")
    print("=" * 78)
    n_gcf = int(group_df['related_lr_accession'].str.startswith('GCF_', na=False).sum())
    n_run = len(group_df) - n_gcf
    print(f"  related_lr_accession type:")
    print(f"    ENA run match (sufficient_for_hybrid on LR row): {n_run:,}")
    print(f"    GCF assembly match (closed RefSeq genome):       {n_gcf:,}")
    print()

    run_rows = group_df[~group_df['related_lr_accession'].str.startswith('GCF_', na=False)]
    vc = run_rows['lr_sufficient_for_hybrid'].value_counts(dropna=False)
    nT = int(vc.get(True, 0)); nF = int(vc.get(False, 0))
    nNA = int(run_rows['lr_sufficient_for_hybrid'].isna().sum())
    print(f"  sufficient_for_hybrid (on the LR row) for ENA-run matches:")
    print(f"    True    : {nT:>5,}")
    print(f"    False   : {nF:>5,}")
    print(f"    missing : {nNA:>5,}  (LR row not found in metadata or column blank)")
    print()

    cov = group_df['lr_est_coverage_5_5Mb'].dropna()
    if len(cov):
        q1, med, q3 = cov.quantile([0.25, 0.5, 0.75])
        print(f"  est_coverage_5_5Mb distribution (n={len(cov):,}):")
        print(f"    min={cov.min():.1f}  Q1={q1:.1f}  median={med:.1f}  Q3={q3:.1f}  max={cov.max():.1f}  IQR={q3-q1:.1f}")
        print(f"    coverage < 10×  : {int((cov < 10).sum()):>5,}  ({(cov<10).mean():.1%})  ('very targeted')")
        print(f"    coverage 10–30× : {int(((cov >= 10) & (cov < 30)).sum()):>5,}  ({((cov>=10)&(cov<30)).mean():.1%})")
        print(f"    coverage ≥ 30×  : {int((cov >= 30).sum()):>5,}  ({(cov>=30).mean():.1%})  ('hybrid-capable')")
        print(f"    coverage ≥ 50×  : {int((cov >= 50).sum()):>5,}  ({(cov>=50).mean():.1%})")
    print()

ai = sub[sub['platform_flagged_lr']]
aii = sub[~sub['platform_flagged_lr']]

_report(ai,  "Ai  — platform already mentioned ONT/PacBio (hybrid candidates)")
_report(aii, "Aii — platform was Illumina-only (novel LR via BioSample lookup)")
_report(sub, "Combined  Ai + Aii")

Ai  — platform already mentioned ONT/PacBio (hybrid candidates)   (n = 1,161)
  related_lr_accession type:
    ENA run match (sufficient_for_hybrid on LR row): 1,161
    GCF assembly match (closed RefSeq genome):       0

  sufficient_for_hybrid (on the LR row) for ENA-run matches:
    True    : 1,030
    False   :   131
    missing :     0  (LR row not found in metadata or column blank)

  est_coverage_5_5Mb distribution (n=1,161):
    min=0.0  Q1=85.1  median=169.6  Q3=305.9  max=2155.9  IQR=220.8
    coverage < 10×  :    26  (2.2%)  ('very targeted')
    coverage 10–30× :    51  (4.4%)
    coverage ≥ 30×  : 1,084  (93.4%)  ('hybrid-capable')
    coverage ≥ 50×  : 1,030  (88.7%)

Aii — platform was Illumina-only (novel LR via BioSample lookup)   (n = 1,070)
  related_lr_accession type:
    ENA run match (sufficient_for_hybrid on LR row): 1,070
    GCF assembly match (closed RefSeq genome):       0

  sufficient_for_hybrid (on the LR row) for ENA-run matches:
    True    :   802
    F